# 22 — Best retrieval, pushed further (honest corpus-grounded evaluation)

This notebook builds on notebook 11 (the highest corpus-grounded scores) and raises the
retrieval ceiling with three changes, while removing a methodological weakness that
inflated the earlier number.

**What is improved**

1. **Cleaner benchmark** — the model-filtered, balanced set from notebook 19
   (\`AWMF_Golden_Dataset_200Q_AIFiltered.csv\`), so every question is genuinely about the
   ten target conditions rather than a keyword coincidence.
2. **Wide retrieve, hard rerank, tight keep** — dense (top 50) and BM25 (top 50) are fused
   with Reciprocal Rank Fusion, reranked by the cross-encoder, and only the strongest few
   chunks are kept. A wider pool lifts recall; a tighter final set lifts precision.
3. **Stronger judge** — the final numbers are scored by Claude-Sonnet (the judge with the
   highest human agreement in the meta-evaluation), not the lighter development judge.

**What is corrected — independent ground truth.** In notebook 11 the corpus-grounded
reference answers were written from the *same* reranked chunks the pipeline is then scored
against, which mechanically inflates recall. Here the reference answers are written from a
*separate, broad* context (the native German question, a larger pool, no reranker, a
different model), so context recall measures whether the evaluated retriever genuinely
surfaces the answer — not whether it recovers text it was handed.

No data is written to the managed database: the BM25 index is built in memory from a
read-only query, and the vector collection is read only.

## 1. Install dependencies

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers rank_bm25 sqlalchemy

## 2. VertexAI import patch

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
for name, attr, val in [("langchain_community.llms","VertexAI",DummyVertexAI),
                        ("langchain_community.chat_models","ChatVertexAI",DummyChatVertexAI),
                        ("langchain_community.chat_models.vertexai","ChatVertexAI",DummyChatVertexAI),
                        ("langchain_community.llms.vertexai","VertexAI",DummyVertexAI)]:
    m = types.ModuleType(name); setattr(m, attr, val); sys.modules[name] = m

## 3. Setup: benchmark, DB, embedder, reranker, BM25, models, prompts

In [ ]:
import os, json, time, random, glob
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'

# Clean benchmark from notebook 19; fall back to the old one if it is not present yet.
_cands = glob.glob('/content/drive/MyDrive/**/AWMF_Golden_Dataset_200Q_AIFiltered.csv', recursive=True) \
      or glob.glob('/content/drive/MyDrive/**/AWMF_Golden_Dataset_200Q_Final.csv', recursive=True)
DATASET_PATH = _cands[0]
df = pd.read_csv(DATASET_PATH)
print("Benchmark:", DATASET_PATH.split('/')[-1], "| rows:", len(df), "| columns:", list(df.columns))

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
COLLECTION = "awmf_baseline_bge"

# Dense store (read-only) + query embedder
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vector_store = PGVector(embeddings=bge, collection_name=COLLECTION, connection=NEON, use_jsonb=True)

# BM25 index, built in memory from a read-only pull of the same chunks (no DB writes)
engine = create_engine(NEON, pool_pre_ping=True)
with engine.connect() as conn:
    rows = conn.execute(text("""SELECT document FROM langchain_pg_embedding
                                WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"""),
                        {"c": COLLECTION})
    all_texts = [r[0] for r in rows]
bm25_index = BM25Okapi([t.lower().split(" ") for t in all_texts])
print("BM25 index built from", len(all_texts), "chunks.")

# Cross-encoder reranker (local, no API cost)
_device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=_device)
print("Reranker device:", _device)

def make_llm(model, max_tokens=4096):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0, max_tokens=max_tokens, max_retries=6)

mistral   = make_llm("mistralai/mistral-large")            # single generator, as in notebook 11
gt_author = make_llm("anthropic/claude-sonnet-4.6", 600)   # writes the independent reference answers

# Retrieval knobs: cast a wide net, keep only the strongest few.
K_DENSE, K_BM25, RRF_K = 50, 50, 60
K_FINAL   = 8      # tighter than notebook 11's 10 -> higher precision
GT_POOL_N = 40     # broad context for the independent ground truth (no reranker)

expansion_prompt = PromptTemplate(template="""You are an expert medical search term generator.
First, translate the following English medical question into German.
Then add 3-4 highly formal German clinical synonyms / related conditions / MeSH terms that would appear in a clinical guideline.
Output ONLY the German question plus the synonyms as a single continuous search string. No labels, no bullets.

English Question:
{question}""", input_variables=["question"])

qa_prompt = PromptTemplate(template="""You are an expert medical AI. Read the German clinical guidelines and answer the medical question in ENGLISH.
Use ONLY the provided German context. If the context does not contain the answer, reply with exactly: NOT_IN_CORPUS

Context (German):
{context}

Question (English):
{question}

Answer (English):""", input_variables=["context", "question"])

gt_author_prompt = PromptTemplate(template="""You are building a reference answer key from official German AWMF guideline excerpts.
Using ONLY the excerpts below, write a concise, factual ENGLISH reference answer to the question.
Do NOT use outside knowledge. If the excerpts do not contain enough information, respond with exactly: NOT_IN_CORPUS

Excerpts (German):
{context}

Question (English):
{question}

Reference answer (English):""", input_variables=["context", "question"])
print("Setup complete.")

## 4. Retrieval — hybrid fusion with a tight, reranked result

\`hybrid_retrieve\` is the *evaluated* pipeline. \`broad_context\` is used only to write the
independent ground truth: it takes a larger pool, uses the native German question, and does
**not** rerank, so it is deliberately different from what the evaluated retriever returns.

In [ ]:
def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try: return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            time.sleep(min(base*(2**a)+random.uniform(0,3), 120))

def expand_query(llm, english_question):
    return safe_invoke(llm, expansion_prompt.format(question=english_question))

def rrf_merge(*lists, k=RRF_K):
    scores, keep = {}, {}
    for lst in lists:
        for rank, t in enumerate(lst):
            key = t[:120]
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            keep.setdefault(key, t)
    return [keep[key] for key in sorted(keep, key=lambda kk: scores[kk], reverse=True)]

def hybrid_retrieve(german_query, k_final=K_FINAL):
    dense = [d.page_content for d in vector_store.as_retriever(search_kwargs={"k": K_DENSE}).invoke(german_query)]
    sparse = bm25_index.get_top_n(german_query.lower().split(" "), all_texts, n=K_BM25)
    fused = rrf_merge(dense, sparse)
    scores = reranker.predict([[german_query, t] for t in fused])
    reranked = [t for _, t in sorted(zip(scores, fused), key=lambda x: x[0], reverse=True)]
    return reranked[:k_final]

def broad_context(german_question, n=GT_POOL_N):
    # Independent of the evaluated retriever: native German query, larger pool, no reranker.
    dense = [d.page_content for d in vector_store.as_retriever(search_kwargs={"k": n}).invoke(german_question)]
    sparse = bm25_index.get_top_n(german_question.lower().split(" "), all_texts, n=n)
    return rrf_merge(dense, sparse)[:n]

# smoke test
_q = df.iloc[0]['English_Open_Question']
_gq = expand_query(mistral, _q)
_ctx = hybrid_retrieve(_gq)
print("Expanded:", _gq[:110], "...")
print("Kept", len(_ctx), "chunks. Top:", _ctx[0][:180], "...")

## 5. Independent corpus-grounded ground truth (one-time)

Reference answers are written from \`broad_context\` (native German question, wide pool, no
reranker), so they are not copied from the evaluated retriever's output. Questions with no
answer in the corpus are marked \`NOT_IN_CORPUS\`; the answerable share is the honest corpus
coverage. Saved under a new filename so the earlier ground truth is untouched.

In [ ]:
GT_FILE = DRIVE_PATH + "AWMF_CorpusGrounded_GT_Independent.csv"
if os.path.exists(GT_FILE):
    gt_rows = pd.read_csv(GT_FILE).to_dict('records'); start = len(gt_rows)
    print(f"Resuming independent GT from {start}/{len(df)}")
else:
    gt_rows, start = [], 0

# A native German question drives the GT retrieval; fall back to the expansion if absent.
def german_query_for(row):
    for c in ("German_Open_Question", "German_Question"):
        if c in row and isinstance(row[c], str) and row[c].strip():
            return row[c]
    return expand_query(gt_author, row['English_Open_Question'])

for i in range(start, len(df)):
    r = df.iloc[i]; q = r['English_Open_Question']
    try:
        ctx = broad_context(german_query_for(r))
        ans = safe_invoke(gt_author, gt_author_prompt.format(context="\n\n".join(ctx), question=q))
        gt_rows.append({"English_Open_Question": q,
                        "medqa_ground_truth": r['English_Correct_Text'],
                        "corpus_ground_truth": ans})
        pd.DataFrame(gt_rows).to_csv(GT_FILE, index=False)
        if (i+1) % 20 == 0:
            cov = sum(1 for x in gt_rows if x['corpus_ground_truth'] != 'NOT_IN_CORPUS') / len(gt_rows)
            print(f"{i+1}/{len(df)}  in-corpus coverage: {cov:.0%}")
        time.sleep(1.2)
    except Exception as e:
        print("err", i, str(e)[:90]); time.sleep(5)

gt_df = pd.read_csv(GT_FILE)
print(f"\nCORPUS COVERAGE: {(gt_df['corpus_ground_truth'] != 'NOT_IN_CORPUS').mean():.0%} answerable from the AWMF Top-10 corpus.")

## 6. Generate answers with the evaluated pipeline (Mistral + hybrid retrieve)

In [ ]:
gt_map = pd.read_csv(GT_FILE).set_index('English_Open_Question')['corpus_ground_truth'].to_dict()
RES_FILE = DRIVE_PATH + "MISTRAL_HYBRID_v22_results.json"
if os.path.exists(RES_FILE):
    res = json.load(open(RES_FILE)); done = set(res["question"])
    print(f"Resuming generation -- {len(done)} done")
else:
    res = {"question": [], "answer": [], "contexts": [], "medqa_ground_truth": [], "corpus_ground_truth": []}; done = set()

work = df.reset_index(drop=True)
print(f"Generating on {len(work)} questions | K_DENSE={K_DENSE} K_BM25={K_BM25} -> RRF -> rerank -> keep {K_FINAL}")
for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx = hybrid_retrieve(expand_query(mistral, q))
        ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))
        res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
        json.dump(res, open(RES_FILE, 'w'))
        if len(res["question"]) % 10 == 0: print(f"{len(res['question'])}/{len(work)}")
        time.sleep(1.2)
    except Exception as e:
        print("err", i, str(e)[:90]); time.sleep(6)
print("Done ->", RES_FILE)

## 7. RAGAS evaluation with the strong judge

Scored by Claude-Sonnet (highest human agreement in the meta-evaluation) for the reported
numbers. Two views: the full set against the MedQA answer key, and the answerable subset
against the independent corpus-grounded key. Context Precision and Context Recall are the
metrics expected to approach 0.9; Answer Relevancy and Faithfulness are bounded lower by
the RAGAS definitions and the deliberate strict-grounding trade-off.

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

# Claude judge for the final numbers; swap to gpt-4o-mini while iterating to save cost.
judge = LangchainLLMWrapper(make_llm("anthropic/claude-sonnet-4.6"))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})
data = json.load(open(RES_FILE))

def eval_with(gt_key, answerable_only=False):
    rows = list(zip(data["question"], data["answer"], data["contexts"], data[gt_key]))
    if answerable_only:
        rows = [r for r in rows if r[3] not in (None, "", "NOT_IN_CORPUS")]
    dd = {"question":[r[0] for r in rows], "answer":[r[1] for r in rows],
          "contexts":[r[2] for r in rows], "ground_truth":[r[3] for r in rows]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return len(rows), out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3)

n, s = eval_with("medqa_ground_truth");            print(f"== vs MedQA GT (n={n}) ==\n{s}\n")
n, s = eval_with("corpus_ground_truth", True);     print(f"== vs INDEPENDENT corpus-grounded GT, answerable (n={n}) ==\n{s}")

## Reading the result

Compare the corpus-grounded row against notebook 11 (0.688 / 0.805). Because the ground
truth here is independent of the retriever, a high Context Recall now means the pipeline
genuinely surfaces the answer rather than recovering handed-in text. If Context Precision
and Context Recall land near 0.85-0.9 on this honest setup, that is the defensible headline.
If they fall short, the levers to try next are a wider pool (K_DENSE / K_BM25 to 80-100), a
tighter keep (K_FINAL to 6), and — the one change that needs a fresh ingestion, so it is
out of scope here — header-aware chunking of the AWMF PDFs.